# TCC: detecÃ§Ã£o de imagens geradas por IA (FSD e UnivFD)
Execute com **Ambiente de execuÃ§Ã£o â†’ Executar tudo**, com GPU T4 ligada. Os resultados sÃ£o salvos no Google Drive em `MyDrive/tcc_detectores`.
Semente fixa = 42. O FSD usa o repositÃ³rio oficial `ductai199x/Forensic-Self-Descriptions-CVPR25` com `max_size=512` e limiar -2,0. O UnivFD usa limiar 0,5.

In [ ]:
# 1) GPU + Drive (o Colab vai pedir autorizaÃ§Ã£o do Drive: clique em permitir)
import subprocess, os
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)
from google.colab import drive
drive.mount('/content/drive')
SAVE = "/content/drive/MyDrive/tcc_detectores"
os.makedirs(SAVE, exist_ok=True)
BASE = "/content/experimento_detectores"
for c in ("real", "fake"): os.makedirs(f"{BASE}/{c}", exist_ok=True)
print("Salvando em", SAVE)

In [ ]:
# 2) InstalaÃ§Ã£o (2 a 3 min)
import subprocess
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True); print((r.stdout + r.stderr)[-1500:])
sh("pip -q install ftfy regex pillow-heif click tqdm huggingface_hub 2>&1 | tail -1")
sh("cd /content && rm -rf UniversalFakeDetect Forensic-Self-Descriptions-CVPR25 && git clone -q https://github.com/Yuheng-Li/UniversalFakeDetect && git clone -q https://github.com/ductai199x/Forensic-Self-Descriptions-CVPR25.git && git -C UniversalFakeDetect log -1 --format='UnivFD %H' && git -C Forensic-Self-Descriptions-CVPR25 log -1 --format='FSD %H'")
sh("pip -q install --no-deps -e /content/Forensic-Self-Descriptions-CVPR25 2>&1 | tail -1")

In [ ]:
# 3) Amostra: 100 reais + 100 IA do Tiny-GenImage (HF: TheKernel01/Tiny-GenImage), semente 42
import io, collections, numpy as np, pandas as pd, pyarrow.parquet as pq
from huggingface_hub import hf_hub_download
from PIL import Image
SEED = 42
p = hf_hub_download("TheKernel01/Tiny-GenImage", "data/validation-00000-of-00004.parquet", repo_type="dataset")
t = pq.read_table(p)
lab = np.array(t.column("label").to_pylist()); gen = np.array(t.column("generator").to_pylist())
rng = np.random.default_rng(SEED)
idx = {"real": rng.choice(np.where(lab == 0)[0], 100, replace=False), "fake": rng.choice(np.where(lab == 1)[0], 100, replace=False)}
rows = []
for cls, ii in idx.items():
    for i in ii:
        b = t.column("image")[int(i)].as_py()["bytes"]
        im = Image.open(io.BytesIO(b)); ext = (im.format or "bin").lower()
        fn = f"{cls}/{cls}_{int(i):05d}.{ext}"
        with open(f"{BASE}/{fn}", "wb") as f: f.write(b)
        rows.append(dict(file=fn, label=int(cls == "fake"), generator=int(gen[i]), fmt=im.format, w=im.width, h=im.height, kb=len(b)/1024))
df = pd.DataFrame(rows)
print(pd.crosstab(df.label, df.fmt), "\n", df.groupby("label")[["w", "h", "kb"]].mean().round(0))
df.to_csv(f"{SAVE}/amostra_meta.csv", index=False)

In [ ]:
# 4) UnivFD (CLIP ViT-L/14): original e com TODAS as imagens recomprimidas em JPEG (controle do viÃ©s PNG x JPEG)
import sys, io, torch, torchvision.transforms as T
sys.path.insert(0, "/content/UniversalFakeDetect"); os.chdir("/content/UniversalFakeDetect")
from models import get_model
um = get_model("CLIP:ViT-L/14")
um.fc.load_state_dict(torch.load("pretrained_weights/fc_weights.pth", map_location="cpu")); um.eval().cuda()
tf = T.Compose([T.CenterCrop(224), T.ToTensor(), T.Normalize([0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711])])
def jpeg(im, q):
    b = io.BytesIO(); im.convert("RGB").save(b, format="JPEG", quality=q); b.seek(0); return Image.open(b)
@torch.no_grad()
def u_score(im): return um(tf(im.convert("RGB")).unsqueeze(0).cuda()).sigmoid().item()
from tqdm.auto import tqdm
df["u_orig"] = [u_score(Image.open(f"{BASE}/{f}")) for f in tqdm(df.file, desc="UnivFD original")]
df["u_j95"] = [u_score(jpeg(Image.open(f"{BASE}/{f}"), 95)) for f in tqdm(df.file, desc="UnivFD jpeg95")]
df["u_j75"] = [u_score(jpeg(Image.open(f"{BASE}/{f}"), 75)) for f in tqdm(df.file, desc="UnivFD jpeg75")]
df.to_csv(f"{SAVE}/resultados.csv", index=False)

In [ ]:
# 5) FSD (max_size=512, limiar -2,0). Salva no Drive a cada condiÃ§Ã£o. Pode levar de 5 a 20 min.
sys.path.insert(0, "/content/Forensic-Self-Descriptions-CVPR25")
from fsd import FSDDetector
det = FSDDetector.load(device="cuda")
det.config["fsd"]["max_size"] = 512
print("limiar:", det.threshold, "| cfg:", det.config["fsd"])
import tempfile
tmp = tempfile.mkdtemp()
def f_run(q, desc):
    zs = []
    for i, fn in enumerate(tqdm(df.file, desc=desc)):
        path = f"{BASE}/{fn}"
        if q:
            path = f"{tmp}/{i}.jpg"; Image.open(f"{BASE}/{fn}").convert("RGB").save(path, format="JPEG", quality=q)
        try: zs.append(float(det.score(path).z_score))
        except Exception as e: print("erro", fn, repr(e)[:80]); zs.append(np.nan)
        torch.cuda.empty_cache()
    return zs
for col, q in [("f_orig", None), ("f_j95", 95), ("f_j75", 75)]:
    df[col] = f_run(q, col)
    df.to_csv(f"{SAVE}/resultados.csv", index=False)
print("FSD concluÃ­do")

In [ ]:
# 6) MÃ©tricas, tabelas, figuras
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, confusion_matrix
def resumo(nome, y, s, pred):                       # s: maior = mais fake
    ok = ~np.isnan(s); y, s, pred = y[ok], s[ok], pred[ok]
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    fpr, tpr, th = roc_curve(y, s); j = np.argmax(tpr - fpr)
    return dict(condicao=nome, n=int(ok.sum()), AUC=roc_auc_score(y, s), AP=average_precision_score(y, s), acuracia=(tp+tn)/len(y),
                precisao=tp/(tp+fp) if tp+fp else np.nan, recall=tp/(tp+fn), especificidade=tn/(tn+fp), TN=tn, FP=fp, FN=fn, TP=tp,
                limiar_youden=th[j], acc_youden=((s >= th[j]) == y).mean())
y = df.label.values; R = []
for k, nome in [("u_orig", "UnivFD original"), ("u_j95", "UnivFD JPEG q95"), ("u_j75", "UnivFD JPEG q75")]:
    R.append(resumo(nome, y, df[k].values, (df[k].values > 0.5).astype(int)))
for k, nome in [("f_orig", "FSD original"), ("f_j95", "FSD JPEG q95"), ("f_j75", "FSD JPEG q75")]:
    if k in df: R.append(resumo(nome, y, -df[k].values, (df[k].values < -2.0).astype(int)))
R = pd.DataFrame(R).round(3); print(R.to_string(index=False)); R.to_csv(f"{SAVE}/resumo_metricas.csv", index=False)

# por gerador (AUC contra as reais)
g = {1: "ADM", 2: "BigGAN", 3: "GLIDE", 4: "Midjourney", 5: "SD14", 6: "SD15", 7: "VQDM", 8: "Wukong"}
rows = []
for gi in sorted(set(df.generator) - {0}):
    m = df[(df.generator == gi) | (df.label == 0)]
    r = dict(gerador=g[gi], n=int((df.generator == gi).sum()), AUC_UnivFD=roc_auc_score(m.label, m.u_orig))
    if "f_orig" in df and m.f_orig.notna().all(): r["AUC_FSD"] = roc_auc_score(m.label, -m.f_orig)
    rows.append(r)
G = pd.DataFrame(rows).round(3); print(G.to_string(index=False)); G.to_csv(f"{SAVE}/auc_por_gerador.csv", index=False)

# figuras: ROC e histogramas
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for k, nome, sgn in [("u_orig", "UnivFD", 1), ("f_orig", "FSD", -1)]:
    if k in df and df[k].notna().all():
        fpr, tpr, _ = roc_curve(y, sgn * df[k]); ax[0].plot(fpr, tpr, label=f"{nome} (AUC={roc_auc_score(y, sgn*df[k]):.2f})")
ax[0].plot([0, 1], [0, 1], "k--"); ax[0].set(title="ROC (imagens originais)", xlabel="Falso positivo", ylabel="Recall (fake)"); ax[0].legend()
for c, lbl in [(0, "real"), (1, "fake")]: ax[1].hist(df[df.label == c].u_orig, bins=20, alpha=.6, label=lbl)
ax[1].axvline(.5, color="k", ls="--", label="limiar 0,5"); ax[1].set(title="Escores UnivFD", xlabel="score"); ax[1].legend()
if "f_orig" in df:
    for c, lbl in [(0, "real"), (1, "fake")]: ax[2].hist(df[df.label == c].f_orig.clip(-25, 2), bins=25, alpha=.6, label=lbl)
    ax[2].axvline(-2, color="k", ls="--", label="limiar -2"); ax[2].set(title="Escores FSD (z)", xlabel="z_score"); ax[2].legend()
plt.tight_layout(); plt.savefig(f"{SAVE}/figuras_roc_hist.png", dpi=200); plt.show()

# McNemar (limiares padrÃ£o, imagens originais)
from scipy.stats import binomtest
if "f_orig" in df:
    uc = (df.u_orig > .5).astype(int) == df.label; fc = (df.f_orig < -2).astype(int) == df.label
    b, c = int((fc & ~uc).sum()), int((~fc & uc).sum())
    print(f"acertos sÃ³ FSD={b}, sÃ³ UnivFD={c}, ambos={int((fc & uc).sum())}, nenhum={int((~fc & ~uc).sum())}; McNemar p={binomtest(min(b, c), b + c, .5).pvalue:.3f}" if b + c else "sem discordÃ¢ncias")
print("Tudo salvo em", SAVE)